In [3]:
import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from data_loader import load_train_data, load_test_data
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import joblib
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [4]:
# 训练集：2022-01~2023-02
X_trains, X_tests, y_trains, y_tests = [], [], [], []
for month in [f"{year}-{str(m).zfill(2)}" for year in range(2022, 2024) for m in range(1, 13)]:
    if month >= "2023-03":
        break
    malicious_output_txt_path = f"/Data2/hxq/MalGuard/fea_ex/dataset/malware_features_{month}.txt"
    benign_output_txt_path = f"/Data2/hxq/MalGuard/fea_ex/dataset/benign_features_{month}.txt"
    X_train, X_test, y_train, y_test = load_train_data(malicious_output_txt_path, benign_output_txt_path)
    X_trains.append(X_train)
    X_tests.append(X_test)
    y_trains.append(y_train)
    y_tests.append(y_test)
X_train = np.vstack(X_trains)
y_train = np.hstack(y_trains)
X_test = np.vstack(X_tests)
y_test = np.hstack(y_tests)

In [5]:
from train_with_lime import train_with_progress_bar, load_sensitive_apis

sensitive_api_file = r"/Data2/hxq/MalGuard/API-call-graph/gpt_prompt_result_closeness.json"
with open(sensitive_api_file, "r") as f:
    sensitive_apis = json.load(f)
sensitive_apis = load_sensitive_apis(sensitive_api_file) 

model_save_path = r"./models"
os.makedirs(model_save_path, exist_ok=True)

# NB
nb_model = GaussianNB()
train_with_progress_bar(nb_model, X_train, y_train, X_test, y_test, "Naive Bayes", model_save_path,
                        sensitive_apis, n_iter=100)

# MLP
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1, warm_start=True, random_state=42)
train_with_progress_bar(mlp_model, X_train, y_train, X_test, y_test, "Multi Layer Perceptron",
                        model_save_path, sensitive_apis, n_iter=500)

# RF
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
train_with_progress_bar(rf_model, X_train, y_train, X_test, y_test, "Random Forest", model_save_path,
                        sensitive_apis, n_iter=100)

# DT
dt_model = DecisionTreeClassifier(random_state=42)
train_with_progress_bar(dt_model, X_train, y_train, X_test, y_test, "Decision Tree", model_save_path,
                        sensitive_apis, n_iter=100)

Training Naive Bayes: 100%|██████████| 100/100 [00:00<00:00, 371.60iter/s]



Naive Bayes Results
              precision    recall  f1-score   support

         0.0    0.56979   0.98810   0.72279       252
         1.0    0.96970   0.33803   0.50131       284

    accuracy                        0.64366       536
   macro avg    0.76975   0.66306   0.61205       536
weighted avg    0.78168   0.64366   0.60543       536

Naive Bayes saved to ./models/naive_bayes/naive_bayes_model.pkl


Training Multi Layer Perceptron: 100%|██████████| 500/500 [00:09<00:00, 53.98iter/s]



Multi Layer Perceptron Results
              precision    recall  f1-score   support

         0.0    0.96047   0.96429   0.96238       252
         1.0    0.96820   0.96479   0.96649       284

    accuracy                        0.96455       536
   macro avg    0.96434   0.96454   0.96443       536
weighted avg    0.96457   0.96455   0.96456       536

Multi Layer Perceptron saved to ./models/multi_layer_perceptron/multi_layer_perceptron_model.pkl


Training Random Forest: 100%|██████████| 100/100 [00:20<00:00,  4.79iter/s]



Random Forest Results
              precision    recall  f1-score   support

         0.0    0.95703   0.97222   0.96457       252
         1.0    0.97500   0.96127   0.96809       284

    accuracy                        0.96642       536
   macro avg    0.96602   0.96674   0.96633       536
weighted avg    0.96655   0.96642   0.96643       536

Random Forest saved to ./models/random_forest/random_forest_model.pkl


Training Decision Tree: 100%|██████████| 100/100 [00:01<00:00, 81.03iter/s]


Decision Tree Results
              precision    recall  f1-score   support

         0.0    0.95635   0.95635   0.95635       252
         1.0    0.96127   0.96127   0.96127       284

    accuracy                        0.95896       536
   macro avg    0.95881   0.95881   0.95881       536
weighted avg    0.95896   0.95896   0.95896       536

Decision Tree saved to ./models/decision_tree/decision_tree_model.pkl


In [6]:
def test_model(model, X_test, y_test, model_name):
    """
    测试模型并打印 precision, recall, f1
    """
    print(f"\n{'='*50}")
    print(f"Testing {model_name}")
    print(f"{'='*50}")

    # 预测
    y_pred = model.predict(X_test)

    # 计算指标
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # 打印结果
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

    # 详细分类报告
    print(f"\n{classification_report(y_test, y_pred, target_names=['Benign', 'Malicious'])}")

    return precision, recall, f1

In [7]:
# 从 models 目录加载已训练的模型并测试
models_dir = r'/Data2/hxq/MalGuard/model_training/models'

results = []

# 可加载的模型列表
model_files = [
    "naive_bayes",
    "decision_tree",
    "random_forest",
    "multi_layer_perceptron",
]

# 测试模型在2022-01~2024-03每个月的数据集上的表现
# 先加载所有模型为全局变量
print("Loading all models...")
models = {}
for model_file in model_files:
    model_path = os.path.join(models_dir, model_file, f"{model_file}_model.pkl")
    if os.path.exists(model_path):
        models[model_file] = joblib.load(model_path)
        print(f"  Loaded: {model_file}")


Loading all models...
  Loaded: naive_bayes
  Loaded: decision_tree
  Loaded: random_forest
  Loaded: multi_layer_perceptron


In [12]:
# 定义月份列表
months = [f"{year}-{str(m).zfill(2)}" for year in range(2022, 2025) for m in range(1, 13)]

# 存储结果的DataFrame
results = []

# 先循环模型，再循环月份
for model_file, model in models.items():
    model_name = model_file.replace("_", " ").title()
    print(f"\n{'#'*20} Testing {model_name} {'#'*20}")
    
    for month in months:
        if month < "2023-03":
            continue  # 跳过2023-03之前的月份
        mal_data_path = f"/Data2/hxq/MalGuard/fea_ex/dataset/malware_features_{month}.txt"
        ben_data_path = f"/Data2/hxq/MalGuard/fea_ex/dataset/benign_features_{month}.txt"

        X_test, y_test = load_test_data(mal_data_path, ben_data_path)
        
        # 预测并计算恶意类（label=1）的指标
        y_pred = model.predict(X_test)
        precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
        recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
        f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        
        results.append({
            "Model": model_name,
            "Month": month,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        })

# 转换为DataFrame
results_df = pd.DataFrame(results)

# 保存结果
results_df.to_csv("/Data2/hxq/MalGuard/model_training/malicious_monthly_results.csv", index=False)
print("\nResults saved to malicious_monthly_results.csv")


#################### Testing Naive Bayes ####################

#################### Testing Decision Tree ####################

#################### Testing Random Forest ####################

#################### Testing Multi Layer Perceptron ####################

Results saved to malicious_monthly_results.csv
